In [ ]:
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
cap = cv2.VideoCapture("../2_pose_extraction/training_vid/forehand_train.mp4")

if cap.isOpened():
    print("Video opened")
    
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    print(f"Total frames: {total_frames}")
    print(f"Fps: {fps}")
else:
    print("Can't open video")

In [20]:
cap = cv2.VideoCapture("../2_pose_extraction/training_vid/forehand_train.mp4")
df = pd.read_csv("../2_pose_extraction/stroke_csv/forehand_landmarks.csv")

frame_count = 0
labels = {}
stroke_window = 30 # each stroke is about 30 frames long
frames_remaining = 0
current_stroke = 'Neutral'

while cap.isOpened():
    sucess, frame =  cap.read()
    
    if not sucess:
        break
    
    #shows the frame in the video
    cv2.imshow("Video" ,frame)
    
    #waits 50 ms between frames, slowing it down
    key = cv2.waitKey(60) & 0xFF
    
    #checks if frame needs to be labeled from previous labels
    if frames_remaining > 0:
        frame_label = current_stroke
        frames_remaining -= 1
    else: 
        frame_label = "Neutral"
        
    #print(f"Capturing {frame_label}")
        
    #stores each frame as a stroke in the dict. default is neutral    
    labels[frame_count] = frame_label
    
    #allows me to mark the beginning of the stroke by pressing the key and the 29 frames following
    if key == ord('q'):
        print("Quit pressed")
        break
    elif key == ord('f'):
        print(f"Forehand at {frame_count}")
        frames_remaining = stroke_window
        current_stroke = labels[frame_count] = "Forehand" #overrides the Neutral frame assignment from above
    elif key == ord('b'):
        print(f"Backhand at {frame_count}")
        frames_remaining = stroke_window
        current_stroke = labels[frame_count] = "Backhand"
    elif key == ord('s'):
        print(f"Serve at {frame_count}")
        frames_remaining = stroke_window
        current_stroke = labels[frame_count] = "Serve"
    elif key == ord('n'):
        print(f"Neutral at {frame_count}")
        frames_remaining = 0
        current_stroke = labels[frame_count] = "Neutral"
    
    frame_count += 1
    
cap.release()
cv2.destroyAllWindows()

#write the labels into the csv
df['label'] = df['frame'].map(labels).fillna('Neutral')
df.to_csv("../2_pose_extraction/stroke_csv/forehand_labeled.csv", index = False)

print(f"Read {frame_count} frames")
print(f"\nLabel Counts:")
print(df['label'].value_counts())

Forehand at 15
Backhand at 49
Quit pressed


C:\Users\guoda\AppData\Local\Temp\ipykernel_6184\3997065778.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['label'] = df['frame'].map(labels).fillna('Neutral')


Read 84 frames

Label Counts:
label
Neutral     2069
Forehand      31
Backhand      31
Name: count, dtype: int64
